# QMMMtools — preparing a QM/MM system for GROMACS + DFTB+

This notebook walks through the whole package: installing it, choosing the QM region,
what happens to the charges and to the QM/MM boundary, what is written out, how to build
and update `dftb_in.hsd`, how to drive everything from the command line, and — in detail —
how to adapt the tables in `QMMMtools.data` to a system that is not a protein.

**What the package does.** It takes a *finished* classical system (a GROMACS `.top` plus
coordinates), cuts a QM region out of it and writes four files that are guaranteed to
describe the same atoms in the same order:

| file | content |
|---|---|
| `qm.top` | QM–QM bonds as connections (`funct 5`), link atoms / charge points as `[ virtual_sites2 ]`, adjusted charges, corrected `[ molecules ]` |
| `qm.gro` | coordinates in the topology's atom order, velocities preserved |
| `qm.ndx` | `[ QM ]` = QM atoms **+ link atoms**, plus `[ freeze ]` and `[ Water_and_ions ]` |
| `dftb_in.hsd` | the DFTB+ / xTB input, QM atoms in `[ QM ]` order |

**What it does *not* do.** It never equilibrates, minimises or re-parametrises anything.
The MM system you feed in is the MM system you get back, minus the bonded terms of the
QM region.

## 0. Installation

Everything installs straight from GitHub. There are two ways to install, and which one you
want depends on how you use the package.

**`pipx` — for the `qmmmtools` command.** pipx puts the executable on your `PATH` inside
its own isolated environment, so nothing is added to your system or conda Python:

```bash
pipx install git+https://github.com/Vetrov-Anton/QMMMtools.git

pipx upgrade QMMMtools
pipx uninstall QMMMtools
```

**`pip` — for `import QMMMtools`.** pipx deliberately hides the package from your
interpreter, so a pipx install does *not* make the import below work. For notebooks and
scripts use pip, into whatever environment you work in:

```bash
pip install git+https://github.com/Vetrov-Anton/QMMMtools.git
```

**From a clone**, if you want to edit the tables in `QMMMtools/data.py` permanently:

```bash
git clone https://github.com/Vetrov-Anton/QMMMtools.git
cd QMMMtools
pip install -e .          # editable
pipx install .            # and/or the command line tool
```

Installing both ways is perfectly reasonable: pipx for the command, pip for the library.

In [ ]:
import QMMMtools
from QMMMtools import data      # every chemistry-specific table lives here

# 'INFO' reports every decision, 'DEBUG' also lists each cut bond and rewritten #include,
# 'WARNING' keeps quiet.
QMMMtools.set_log_level('INFO')

print('QMMMtools', QMMMtools.__version__)

## 1. Quick start

Five lines. `job()` does the whole preparation, `make_hsd()` writes the QM input,
`check_consistency()` verifies that the output files agree.

The example system is a pyruvate-dehydrogenase-like enzyme: protein + thiamine
pyrophosphate (TPP) + a phosphonate ligand (API) + Mg²⁺ + TIP3P water and ions. The seeds
below are the Mg²⁺ ion, atoms of the two cofactors, and the side chains that coordinate them.

In [ ]:
GRO, TOP = 'npt4.gro', 'topol.top'
SKPATH   = './3ob-3-1-ophyd/'     # Slater–Koster files, as mdrun will see them

qm = QMMMtools.QM(GRO, TOP, 'qm.gro', 'qm.top', 'qm.ndx')

qm.choose_qm_to_extend('@21313,21391,21408,2572,3009,3991,1455,4052,1070,18202,17217,16712')
qm.job()

qm.make_hsd('dftb_in.hsd', method='dftb3-d4', skpath=SKPATH, mixer='anderson')
qm.check_consistency('dftb_in.hsd')

Everything the run decided is available afterwards:

In [ ]:
print('QM atoms (incl. link atoms):', len(qm.qm.atoms))
print('real QM atoms             :', len(qm.qm_idx))
print('link atoms                :', len(qm.la_idx))
print('charge points             :', len(qm.cp_idx))
print('QM charge from the FF     :', round(qm.qm_charge, 4))
print('integer charge used       :', qm.aim_qm_charge)
print('total charge of the output:', round(qm.full_charge(), 6))
print()
print('boundary bonds:')
for bond in qm.qmmm_bonds:
    a, b = bond.atom1, bond.atom2
    print(f'   {a.residue.name}{a.residue.number}:{a.name} -- {b.residue.name}{b.residue.number}:{b.name}')

## 2. Choosing the QM region

Two selectors. Both take [Amber masks](https://parmed.github.io/ParmEd/html/amber.html)
in the numbering of the **input** system, both may be called any number of times, and
their results are unioned.

### 2.1 `choose_qm_to_extend` — grow to the breakable bonds

Starts a depth-first walk at every atom the mask selects and follows bonds until it would
step over a *breakable* bond. This is how you say "this whole side chain" or "this whole
cofactor" while giving only one atom.

`qm.breakable_bonds` is a set of **directed** `(QM atom name, MM atom name)` pairs.
`('CB', 'CA')` means *walking from an atom named CB onto one named CA is a cut*. The
direction matters: a walk that arrives at CB stops there, but a walk that starts at CA
still reaches CB and the rest of the side chain.

In [ ]:
print('default (protein) cuts:')
for pair in sorted(qm.breakable_bonds):
    print(f'   {pair[0]:>4s} -> {pair[1]:<4s}   link atom at {qm.H_dist[pair]:.2f} A from {pair[0]}')

`extend_until_break` can be called on its own, without adding anything to the selection —
handy for seeing what a seed would pull in before committing to it:

In [ ]:
probe = QMMMtools.QM(GRO, TOP, '/tmp/x.gro', '/tmp/x.top', '/tmp/x.ndx')
grown = probe.extend_until_break('@1070')          # NE2 of a histidine
print(grown)
print()
for atom in probe.itop.view[grown].atoms:
    print(f'  {atom.idx+1:6d}  {atom.residue.name}{atom.residue.number}:{atom.name}')

### 2.2 `choose_qm_manually` — exactly these atoms

No growth at all. Use it for ions, for a hand-picked set, or for distance-based masks.

### 2.3 Combining, and reusing what is already selected

`qm.qm_input_mask` always holds everything selected so far, in input numbering, so later
masks can refer to it. This is the idiomatic way to add a solvation shell around the QM
region *after* the region has been defined:

In [ ]:
qm2 = QMMMtools.QM(GRO, TOP, 'qm2.gro', 'qm2.top', 'qm2.ndx')

qm2.choose_qm_to_extend('@21313,21391,2572,3009')   # grown fragments
qm2.choose_qm_manually('@21408')                    # the Mg2+ ion, as it is
qm2.choose_qm_manually(f'(({qm2.qm_input_mask})<:3.5)&(:SOL)')   # + all water within 3.5 A

print('extend part :', qm2.qm_extend_mask[:60], '...')
print('manual part :', qm2.qm_manual_mask[:80], '...')
print('combined    :', qm2.qm_input_mask[:60], '...')

`<:` selects *whole residues* within a distance, which is what you want for water: a
half-selected water molecule cannot be handled consistently and the module warns about it.

### 2.4 Looking at the region before committing

`determine_qm()` performs only the selection and the system split, so you can inspect the
result before running the rest of `job()`.

In [ ]:
qm2.determine_qm()

from collections import Counter
counts = Counter(a.residue.name for a in qm2.qm.atoms)
print('QM atoms per residue type:', dict(counts))
print('QM moleculetype :', len(qm2.qm_mol.atoms), 'atoms')
print('untouched rest  :', len(qm2.rest.atoms), 'atoms')
print('molecules pulled out of [ molecules ]:', qm2.count)

## 3. Charge

A QM code needs an **integer** charge, while the force-field charges of the selected atoms
almost never sum to one. `job()` without `qm_aim_charge` works it out as

```
aim = round(q_total) − round(q_total − q_QM, threshold=0.25)
```

Ordinary rounding switches over at a fractional part of 0.5. Here the threshold is **0.25**,
because a cut that runs through a charge group leaves part of a formal charge behind on the
MM side, so the MM sum comes out too small in magnitude:

| MM sum | 0.25 (default) | 0.5 (`'nearest'`) | 0.0 (`'away'`) |
|---:|---:|---:|---:|
| 1.43 | **2** | 1 | 2 |
| −1.43 | **−2** | −1 | −2 |
| 0.04 | **0** | 0 | 1 |
| 2.00 | **2** | 2 | 2 |

Any threshold between 0 and 1 works (`charge_rounding=0.4`), and `'nearest'` / `'away'` are
names for 0.5 and 0.0.



The MM sum is rounded **away from zero** by default (`1.43 → 2`, `−1.43 → −2`): a cut that
runs through a charge group leaves part of a formal charge behind on the MM side, so the MM
sum tends to come out too small in magnitude. `charge_rounding='nearest'` restores ordinary
rounding. Careful with a nearly neutral QM region — away-from-zero turns an MM sum of
`−0.04` into `−1`, which is wrong; whenever the two rules disagree both candidates are
logged, so you always see it.

(the whole system and the MM part both carry integer charge, so this is the integer the QM
region must have), and then moves the difference onto the MM atoms so that the total charge
of the written system is exactly what it was.

**Who may accept the charge.** Only protein and nucleic-acid atoms — the residue names in
`qm.redistr_residues`. Never water, never ions: a redistributed water would no longer be
the water model you parametrised. If the MM part is far from an integer you get a warning;
that usually means a cut runs through a charge group and you should either move the
boundary or pass the charge explicitly.

In [ ]:
qm3 = QMMMtools.QM(GRO, TOP, 'qm3.gro', 'qm3.top', 'qm3.ndx')
qm3.choose_qm_to_extend('@3009')            # one asparagine side chain
qm3.determine_qm()
qm3.calculate_charge_qm()

print('FF charge of the QM atoms :', round(qm3.qm_charge, 4))
print('charge of the whole system:', round(qm3.total_charge, 4))
print('integer charge to be used :', qm3.detect_qm_charge())

To override it, pass an explicit value — `job(qm_aim_charge=-2)`, or `--charge -2` on the
command line. Do that whenever you know the chemistry better than the force field does
(a deprotonated phosphate, a metal whose FF charge is scaled, and so on).

**The value must match `QMcharge` in the `.mdp` and `Charge` in `dftb_in.hsd`**;
`make_hsd` takes care of the latter and the CLI reminds you about the former.

**When the threshold changes the answer.** Preparation then warns *"the QM charge is
ambiguous"* and reports both candidates, because the wrong one can keep the QM code from
converging at all. Measured on this very system: the force-field sum was −1.4453; ordinary
rounding gives −1, which never converged the SCC, the 0.25 threshold gives −2, which
converged in 157 iterations. Trying the other candidate costs one command — no
re-preparation:

```bash
qmmmtools rewrite-hsd dftb_in.hsd --charge -1      # and QMcharge = -1 in the .mdp
```

```python
qm.rewrite_hsd('dftb_in.hsd', charge=-1)
```

## 4. The QM/MM boundary: link atoms

Every QM–MM bond gets a hydrogen link atom, written as a two-body virtual site: it sits on
the QM–MM axis at `H_dist` from the QM atom, so its construction weight is
`d / |r_QM−MM|` and it follows the two real atoms during the run.

`H_dist` is the equilibrium **X–H bond length of the QM atom being capped** — C 1.09 Å,
N 1.01 Å, O 0.97 Å, S 1.34 Å. Using anything else stretches or compresses a real chemical
bond inside the QM calculation, and that shows up directly in the QM energy and forces.

With `link_la_to_mm1=True` (the default) a `funct 5` bond is also added between the link
atom and the MM atom. That bond carries no potential; it exists so that grompp generates
the exclusions around the link atom.

In [ ]:
print('[ virtual_sites2 ] rows written for the link atoms:')
print('   site   from     to  funct  weight')
for row in qm.vs2[:5]:
    print('  ', '  '.join(f'{x:>5s}' for x in row[:5]), row[5])

## 5. What happens to the MM boundary charge — `redistr_scheme`

The MM atom at the boundary sits right next to the link atom, and its full MM charge
polarises the QM density unphysically. Five options:

| value | effect |
|---|---|
| `'no'` (default) | leave it alone |
| `'amber'` | zero it, spread the charge over the MM acceptors |
| `'RC'` | zero it, put `q/n` on the midpoint of each MM1–MM2 bond |
| `'RCD'` | as RC but `2q/n` on the midpoint and `−q/n` on MM2 — preserves the dipole |
| `'CS'` | charge shift: `q/n` onto MM2 plus a `+q/n` / `−q/n` pair around it |

`RC`, `RCD` and `CS` add massless charge points as further `[ virtual_sites2 ]` entries.
They are *not* part of the `[ QM ]` group — they are MM point charges.

In [ ]:
for scheme in ('no', 'amber', 'RC', 'RCD', 'CS'):
    trial = QMMMtools.QM(GRO, TOP, '/tmp/s.gro', '/tmp/s.top', '/tmp/s.ndx')
    trial.choose_qm_to_extend('@3009')
    QMMMtools.set_log_level('WARNING')
    trial.job(redistr_scheme=scheme)
    QMMMtools.set_log_level('INFO')
    print(f'{scheme:>6s}: {len(trial.cp_idx):2d} charge points, '
          f'total charge {trial.full_charge():+.6f}')

## 6. MM bonded terms inside the QM region — `mm_retention`

The bonds, angles and dihedrals inside the QM region are described by the QM calculation,
so their MM counterparts must not be counted a second time.

**grompp of this GROMACS build already does this itself** and prints a table of what it
removed, which is why the default is `'no'`. Use the other values only when you want the
cleaned topology on disk — note that they then also pre-empt `GMX_QMMM_BONDED_SCHEME`.

| value | effect |
|---|---|
| `'no'` (default) | leave everything, let grompp do it |
| `'classic'` | grompp's own rule: a term goes once all but one of its atoms are QM, plus the 1-4 pairs between a QM atom and a bonded MM atom |
| `'amber'` | only terms whose atoms are *all* QM |

One thing is **always** done, whatever `mm_retention` says: QM–QM bonds are converted to
`funct 5` (a "connection": no potential, but exclusions are still generated). This is not
cosmetic. Left as `funct 1` they are turned into constraints by `constraints = h-bonds`
*before* grompp removes the QM bonded terms, and the QM hydrogens end up rigid — 50 degrees
of freedom silently disappeared in a measured test.

In [ ]:
for mode in ('no', 'classic', 'amber'):
    trial = QMMMtools.QM(GRO, TOP, '/tmp/m.gro', '/tmp/m.top', '/tmp/m.ndx')
    trial.choose_qm_to_extend('@3009')
    QMMMtools.set_log_level('WARNING')
    trial.job(mm_retention=mode)
    QMMMtools.set_log_level('INFO')
    print(f'{mode:>8s}: {len(trial.qm_mol.angles):6d} angles, '
          f'{len(trial.qm_mol.dihedrals):6d} dihedrals, {len(trial.qm_mol.adjusts):6d} 1-4 pairs')

## 7. Running GROMACS

```bash
gmx grompp -f qm.mdp -c qm.gro -p qm.top -n qm.ndx -o qm.tpr

GMX_QMMM_NREXCL=3 GMX_QMMM_VARIANT=1 \
gmx mdrun -deffnm qm
```

with an `.mdp` that contains

```
QMMM      = yes
QMMM-grps = QM
QMmethod  = RHF        ; grompp insists on a value, this build ignores it
QMbasis   = STO-3G     ; likewise
QMcharge  = -2         ; must equal Charge in dftb_in.hsd
QMmult    = 1
```

`dftb_in.hsd` has to sit in the run directory — mdrun reads it, then overwrites the
coordinates from the trajectory each step. So the *numbers* in `Geometry` do not matter for
the run, but the atom **count** and **order** do.

Read grompp's QM/MM table in the output; it tells you exactly which bonded terms it removed
and how many connections it found already present.

## 8. QM methods

All of the entries below were checked against DFTB+ 21.2; GFN2-xTB was additionally
verified to run through the GROMACS interface.

In [ ]:
QMMMtools.list_methods()

In [ ]:
# DFTB methods need Slater–Koster files, xTB methods do not
qm.make_hsd('dftb_in_d3h5.hsd', method='dftb3-d3h5', skpath=SKPATH, mixer='anderson')
qm.make_hsd('dftb_in_xtb.hsd',  method='gfn2-xtb')

print(open('dftb_in_xtb.hsd').read().split('Analysis')[0][-260:])

### SCC convergence

Charged metal sites are hard for DFTB3. In this very system the SCC does **not** converge
with the DFTB+ default Broyden mixer, not even with 500 iterations; a small-step Anderson
mixer converges it in 62. That is what `mixer='anderson'` writes:

```
Mixer = Anderson {
  MixingParameter = 0.05
  Generations = 8
}
```

`max_scc_iterations` and `scc_tolerance` are separate arguments, and `mixer=` also accepts
a raw HSD string if you want something else.

## 9. Updating an existing `dftb_in.hsd` — `rewrite_hsd`

`make_hsd` writes a file from scratch. `rewrite_hsd` **edits one**: everything it is not
asked to change stays byte for byte, so hand-tuned settings survive. This is the function
to use when the Hamiltonian part was tuned once and only the QM region moved.

In [ ]:
import shutil
shutil.copy('dftb_in.hsd', 'tuned.hsd')

# pretend we tuned something by hand
text = open('tuned.hsd').read().replace('SCCTolerance = 1e-6', 'SCCTolerance = 1e-8')
open('tuned.hsd', 'w').write(text)

qm.rewrite_hsd('tuned.hsd')                     # coordinates only

print('hand edit survived:', 'SCCTolerance = 1e-8' in open('tuned.hsd').read())

In [ ]:
# ... or change the settings as well
qm.rewrite_hsd('tuned.hsd', charge=-3, skpath='/scratch/sk/3ob-3-1/',
               max_scc_iterations=400, mixer='anderson')

# ... or swap the whole Hamiltonian for another method
qm.rewrite_hsd('tuned.hsd', method='gfn2-xtb', charge=-2)

# ... or add / drop any other top-level block
qm.rewrite_hsd('tuned.hsd', blocks={'Driver': 'Driver = {}'})
qm.rewrite_hsd('tuned.hsd', blocks={'Driver': None})       # None deletes it

# geometry=False changes only the settings and leaves the coordinates alone
qm.rewrite_hsd('tuned.hsd', geometry=False, scc_tolerance='1e-7')

### 9.1 Without a topology — only `.gro` + `.ndx`

You do not need the `QM` object at all. `read_qm_geometry` pulls the `[ QM ]` group out of
an index file and the coordinates out of any file ParmEd can read, which is enough to
refresh a `.hsd` — including files somebody else produced.

In [ ]:
geometry = QMMMtools.read_qm_geometry('qm.gro', 'qm.ndx', group='QM')
print(len(geometry), 'atoms |', geometry.type_names)

QMMMtools.rewrite_hsd('tuned.hsd', geometry=geometry, method='dftb3-d4',
                      charge=-2, skpath=SKPATH, mixer='anderson')

Two things to know about this mode.

**Elements.** A `.gro` carries no atomic numbers, so they are inferred. First choice is the
`TypeNames` of the file being rewritten (whenever the atom count matches); otherwise they
are guessed from the atom and residue names. The guess is name-aware — `CA` inside `ALA` is
a carbon, `CA` inside a residue called `CA` is calcium, `SE` in `MSE` is selenium,
`LA`/`CP` are the package's own massless sites. When both sources are available and they
disagree you get a warning naming the atoms; that check has already caught a stale `.hsd`
that belonged to an older selection.

You can also state them outright:

```python
QMMMtools.read_qm_geometry('qm.gro', 'qm.ndx', elements=['C', 'H', 'H', ...])
```

**Precision.** A `.gro` stores 0.001 nm, so coordinates recovered from one are accurate to
0.01 Å. Irrelevant for a QM/MM start, since mdrun overwrites them every step anyway.

In [ ]:
print(data.guess_element('CA', 'ALA'),   # alpha carbon
      data.guess_element('CA', 'CA'),    # calcium ion
      data.guess_element('HG', 'CYS'),   # gamma hydrogen
      data.guess_element('SE', 'MSE'),   # selenomethionine
      data.guess_element("C1'", 'DA'),   # nucleic acid sugar
      data.guess_element('LA', 'XXX'))   # this package's link atom

## 10. The command line

Everything above is also reachable without writing Python. After `pipx install QMMMtools`
the `qmmmtools` command is on your `PATH`.

```bash
qmmmtools prepare \
    -c npt4.gro -p topol.top \
    -e '@21313,21391,21408,2572,3009' \
    --solvate 3.0 \
    --redistr-scheme RCD \
    --hsd --skpath ./3ob-3-1-ophyd/ --mixer anderson
```

| flag | meaning |
|---|---|
| `-e/--extend MASK` | grow the mask to the breakable bonds (repeatable) |
| `-s/--select MASK` | take the mask as it is (repeatable) |
| `--solvate R` | add every water residue within R Å of what is selected so far |
| `--charge N` | explicit QM charge (default: derived from the force field) |
| `--mm-retention`, `--redistr-scheme`, `--no-link-bond` | as the Python arguments |
| `--preset protein\|nucleic\|lipid` | which breakable-bond table to start from (repeatable) |
| `--breakable CG:CB:1.09`, `--no-breakable C:CA` | edit the table from the command line |
| `--hsd [FILE]`, `--method`, `--skpath`, `--mixer`, ... | also write the DFTB+ input |
| `-d/--outdir` | where the default-named outputs go |

The other subcommands:

```bash
qmmmtools rewrite-hsd dftb_in.hsd -c qm.gro -n qm.ndx --charge -2 --mixer anderson
qmmmtools check       -c qm.gro -n qm.ndx --hsd dftb_in.hsd
qmmmtools methods
qmmmtools tables bonds
```

`qmmmtools <subcommand> --help` lists every option; `-v` and `-q` change the log level.

In [ ]:
# the CLI is importable too, so it can be driven from a notebook
from QMMMtools.cli import main
main(['tables', 'bonds'])

## 11. Adapting `QMMMtools.data`

This is the module to touch when the system is not a protein, when a force field uses
unusual names, or when a new element or QM method is needed. **Nothing has to be edited
permanently**: every table is copied onto the `QM` object in `__init__`, so you can
override it per object at run time. Edit `QMMMtools/data.py` itself only for changes you
want in *every* future run — and remember that a `pipx`-installed copy lives inside the
pipx venv, so for permanent edits work from a clone with `pip install -e .`.

### 11.1 Link-atom tables — the two you will actually change

`breakable_bonds` decides *where* the QM region may be cut, `H_dist` decides how far from
the QM atom the capping hydrogen goes. Ready-made sets are shipped for proteins, nucleic
acids and (as a starting point) lipids.

In [ ]:
print('protein :', sorted(data.PROTEIN_BREAKABLE_BONDS))
print('nucleic :', sorted(data.NUCLEIC_BREAKABLE_BONDS))
print('lipid   :', sorted(data.LIPID_BREAKABLE_BONDS))
print()
print('nucleic H_dist:')
for pair, d in sorted(data.NUCLEIC_H_DIST.items()):
    print(f'   {pair[0]:>5s} -> {pair[1]:<5s} {d:.2f} A')

**Replacing a whole set** (run time, per object):

```python
qm.breakable_bonds = set(data.NUCLEIC_BREAKABLE_BONDS)
qm.H_dist          = dict(data.NUCLEIC_H_DIST)
```

**Adding one cut:**

```python
qm.breakable_bonds.add(('CG', 'CB'))
qm.H_dist[('CG', 'CB')] = 1.09        # C–H
```

**Removing a cut you do not want** — e.g. keeping the whole backbone quantum instead of
cutting the peptide bond:

```python
qm.breakable_bonds.discard(('C', 'CA'))
qm.breakable_bonds.discard(('N', 'CA'))
qm.H_dist.pop(('C', 'CA'), None)      # pop(..., None) never raises
qm.H_dist.pop(('N', 'CA'), None)
```

**Mixing sets** — a protein–DNA complex needs both:

```python
qm.breakable_bonds = set(data.PROTEIN_BREAKABLE_BONDS) | set(data.NUCLEIC_BREAKABLE_BONDS)
qm.H_dist          = {**data.PROTEIN_H_DIST, **data.NUCLEIC_H_DIST}
```

**Permanently**, by editing `QMMMtools/data.py`: add or delete entries of
`PROTEIN_BREAKABLE_BONDS` / `PROTEIN_H_DIST`, or define a new pair of tables next to them
(`MY_BREAKABLE_BONDS`, `MY_H_DIST`) and refer to it the same way. Keep the two in step —
a bond in `breakable_bonds` with no `H_dist` entry falls back to the element default and
warns; the reverse is simply unused.

Set `qm.strict_h_dist = True` to turn that warning into an error.

In [ ]:
demo = QMMMtools.QM(GRO, TOP, '/tmp/d.gro', '/tmp/d.top', '/tmp/d.ndx')

demo.breakable_bonds = {('CG', 'CB')}     # cut one bond further out than usual
demo.H_dist = {}                          # ... and "forget" the distance
demo.choose_qm_to_extend('@3009')
demo.job()                                # -> warning, element default used

### 11.2 Residue-name tables

Two sets, both per-object attributes:

* `qm.solvent_and_ions` — residues that may **never** accept redistributed charge, and that
  are reported separately when they end up in the QM region. Already covers
  `SOL/HOH/WAT/TIP3/TIP4/SPC/SPCE/OPC/...` and the usual ion spellings.
* `qm.redistr_residues` — residues that **may** accept it. Defaults to the standard amino
  acids and nucleotides, including protonation variants (`HID/HIE/HIP/HSD/...`) and
  `N`/`C` terminal spellings.

```python
qm.solvent_and_ions.add('T4P')                  # an unusual water name
qm.redistr_residues = set(data.NUCLEIC_ACIDS)   # DNA only
qm.redistr_residues |= {'LIG'}                  # let a ligand take charge too
qm.redistr_residues.discard('PRO')              # keep prolines out of it
```

If nothing is left to accept charge the module falls back to every non-solvent MM atom and
says so; if even that is empty it raises rather than quietly skipping the step.

In [ ]:
print('waters recognised :', sorted(data.WATER_RESIDUES)[:12], '...')
print('ions recognised   :', sorted(data.ION_RESIDUES)[:12], '...')
print('acceptors, default:', len(data.POLYMER_RESIDUES), 'residue names')

### 11.3 Elements

Elements come from the atomic number ParmEd reads out of the `at.num` column of
`[ atomtypes ]`. That is always right, so the fall-back tables are rarely needed:

* `data.TYPE2ELEMENT` — for force-field types with no usable atomic number (massless sites,
  hand-made types). Add an entry if you hit `cannot tell the element of atom …`; fixing
  `at.num` in the force field is better.
* `data.SPECIAL_SITE_ELEMENTS` — virtual sites recognised by atom name in the `.gro`-only path.

```python
data.TYPE2ELEMENT['MYTYPE'] = 'Se'          # module-wide, for this session
data.SPECIAL_SITE_ELEMENTS['DRUD'] = 'H'    # Drude particle in a .gro
```

### 11.4 DFTB parameters for a new element

A DFTB method needs an angular momentum and (for DFTB3) a Hubbard derivative per element.
Both raise with the element name when they are missing, so you know exactly what to add:

```python
data.MAX_ANGULAR_MOMENTUM['Se'] = 'd'
data.HUBBARD_DERIVS['Se']       = -0.11
```

Take the values from the documentation of the Slater–Koster set you use — 3ob and mio
publish them, and they are *not* interchangeable between sets. To *remove* support for an
element (so a typo fails loudly instead of silently producing a bad Hamiltonian):

```python
data.MAX_ANGULAR_MOMENTUM.pop('Li', None)
data.HUBBARD_DERIVS.pop('Li', None)
```

### 11.5 Adding or removing a QM method

`data.QM_METHODS` maps a name to a `QMMethod`. Everything about a method is the HSD text it
contributes, so adding one is a few lines. Check the result by running `dftb+` on the
generated file once — a typo in an HSD block is caught immediately.

In [ ]:
data.QM_METHODS['dftb3-d3zero'] = data.QMMethod(
    name='dftb3-d3zero',
    kind='dftb',                     # 'dftb' needs Slater–Koster files, 'xtb' does not
    description='DFTB3/3ob with plain D3 zero damping',
    third_order=True,                # writes ThirdOrderFull + HubbardDerivs
    hcorrection='HCorrection = Damping {\n      Exponent = 4.05\n    }',
    dispersion=('Dispersion = DftD3 {\n'
                '      Damping = ZeroDamping {\n'
                '        sr6 = 1.25\n'
                '        alpha6 = 29.61\n'
                '      }\n'
                '      s6 = 1.0\n'
                '      s8 = 0.49\n'
                '    }'),
)

QMMMtools.list_methods()
qm.make_hsd('dftb_in_new.hsd', method='dftb3-d3zero', skpath=SKPATH)

An xTB-style entry is even shorter:

```python
data.QM_METHODS['gfn0-xtb'] = data.QMMethod(
    'gfn0-xtb', 'xtb', 'GFN0-xTB via tblite', xtb_method='GFN0-xTB')
```

**Removing** a method you do not want offered:

```python
data.QM_METHODS.pop('dftb2', None)
```

Anything the registry cannot express goes in through `extra=` (appended verbatim) or
through `rewrite_hsd(blocks={...})`.

## 12. A non-protein system, start to finish

The only difference from section 1 is the two tables. This cell is illustrative — it needs
a DNA system to run.

In [ ]:
# --- illustrative, needs dna.gro / dna.top -------------------------------
# qm = QMMMtools.QM('dna.gro', 'dna.top', 'qm.gro', 'qm.top', 'qm.ndx')
#
# qm.breakable_bonds  = set(data.NUCLEIC_BREAKABLE_BONDS)
# qm.H_dist           = dict(data.NUCLEIC_H_DIST)
# qm.redistr_residues = set(data.NUCLEIC_ACIDS)
# qm.strict_h_dist    = True          # refuse to guess a link-atom distance
#
# qm.choose_qm_to_extend('@1234')     # one atom of a base -> the whole base
# qm.job(redistr_scheme='RCD')
# qm.make_hsd('dftb_in.hsd', method='dftb3-d4', skpath=SKPATH)
# qm.check_consistency('dftb_in.hsd')
#
# the same from the command line:
#   qmmmtools prepare -c dna.gro -p dna.top --preset nucleic \
#       -e '@1234' --redistr-scheme RCD --hsd --skpath ./3ob-3-1/

For a lipid, `data.LIPID_BREAKABLE_BONDS` is only a starting point for CHARMM-style
glycerol/ester names — lipid force fields disagree about atom naming more than any other
class, so expect to write the pairs yourself. The procedure is the same: list the bonds you
want cut, give each an X–H distance, done.

## 13. Consistency and troubleshooting

`check_consistency` re-reads the files that were written and verifies that `[ QM ]`, the
`.gro` and the `.hsd` describe the same atoms in the same order. A silent reordering
between these three is the failure mode that is hardest to spot afterwards, so run it
before every production job.

In [ ]:
qm.check_consistency('dftb_in.hsd')

| symptom | cause and fix |
|---|---|
| `SCC is NOT converged` in mdrun | charged metal site; use `mixer='anderson'` (more iterations alone does not help) |
| `No default Bond types` from grompp | a ligand lost its inline bonded parameters; only QM–QM bonds should have been stripped |
| `cannot tell the element of atom …` | `at.num` missing in `[ atomtypes ]`; fix the force field or extend `TYPE2ELEMENT` |
| `[ molecules ] accounts for N atoms but the structure has M` | topology and coordinates are not the same system |
| `these residues are split between the QM moleculetype and the rest` | a molecule is half in, half out — select whole residues (`<:` masks do that) |
| `no link-atom distance for the boundary bond …` (warning) | add the pair to `H_dist`, or accept the element default |
| `mask '…' selects no atoms` | the mask is valid but empty — check the numbering (Amber masks are 1-based) |
| `import QMMMtools` fails after `pipx install` | expected: pipx isolates the package. Use `pip install git+https://github.com/Vetrov-Anton/QMMMtools.git` for library use |

## 14. API summary

**Selection** — `choose_qm_to_extend`, `choose_qm_manually`, `extend_until_break`,
`determine_qm`; attributes `qm_input_mask`, `qm_mask`, `qm_idx`, `qm`, `qm_mol`, `rest`.

**Charge** — `calculate_charge_qm`, `detect_qm_charge`, `redistribute_charge_from_qm_to_mm`,
`full_charge`; attributes `qm_charge`, `total_charge`, `aim_qm_charge`.

**Boundary** — `find_qmmm_bonds`, `vs2_and_LA`, `redistribute_boundary_charge`,
`amber_redist`, `RC_redist`, `RCD_redist`, `CS_redist`; attributes `qmmm_bonds`,
`mm1_atoms`, `vs2`, `la_idx`, `cp_idx`.

**Topology** — `process_bonds`, `process_mm_terms`, `write_ndx`, `write_otop`, `write_ogro`,
`write_outputs`.

**QM input** — `qm_geometry`, `make_hsd`, `rewrite_hsd`, `check_consistency`; module level:
`read_qm_geometry`, `read_index_file`, `write_hsd`, `rewrite_hsd`, `hamiltonian_block`,
`list_methods`, `get_method`, `QMGeometry`, `HsdFile`.

**Everything at once** — `job(qm_aim_charge=None, mm_retention='no', redistr_scheme='no',
link_la_to_mm1=True)`, or `qmmmtools prepare` on the command line.